# STEP 7: LangGraph에서 실제 OpenDART 조회 Node 실행

이 노트북에서는 기존 `src/dart/client.py`의 검증된 OpenDART 호출 함수를 LangGraph Node 안에서 실행합니다.

`기업명·사업연도 State → OpenDART 조회 Node → 요약 Node → 최종 State`

이번 단계에는 LLM, LangChain Tool, 조건 분기, Agent가 없습니다. Node는 Tool이 아니어도 실제 Python 함수와 외부 API를 실행할 수 있다는 점을 확인합니다.

## 0. 이번 Node와 STEP 4 Tool의 차이

STEP 4에서는 LLM이 OpenDART Tool 실행을 요청했고, Python이 Tool을 실행했습니다. 이번에는 LLM이 없습니다. LangGraph의 고정 Edge가 `fetch_financials` Node를 실행합니다.

둘 다 같은 `get_financial_summary()` 함수를 사용하지만, **실행을 결정하는 주체**가 다릅니다.

## 1. 프로젝트 경로와 OpenDART API 키 준비

API 키는 `.env`에서만 읽고 출력하지 않습니다.

In [ ]:
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

current_dir = Path.cwd().resolve()
project_root = current_dir.parent if current_dir.name == "notebooks" else current_dir
sys.path.insert(0, str(project_root))

load_dotenv(project_root / ".env")
api_key = os.getenv("OPENDART_API_KEY")

if not api_key or api_key == "your_opendart_api_key":
    raise ValueError(".env의 OPENDART_API_KEY를 실제 API 키로 바꿔 주세요.")

print("프로젝트 경로와 OpenDART API 키를 준비했습니다.")

## 2. LangGraph와 기존 OpenDART 함수 불러오기

`get_financial_summary()`가 실제 기업명 → `corp_code` → 재무제표 조회를 담당합니다. 이 notebook은 해당 호출 로직을 다시 작성하지 않고 import합니다.

In [ ]:
from typing import TypedDict

from langgraph.graph import END, START, StateGraph

from src.dart.client import get_financial_summary

## 3. State 정의

시작 State에는 기업명과 사업연도만 있습니다. `fetch_financials` Node가 실제 API 결과를 `financial_data`에 추가하고, 다음 Node가 이를 읽어 `summary`를 추가합니다.

In [ ]:
class OpenDARTWorkflowState(TypedDict, total=False):
    company_name: str
    business_year: int
    financial_data: dict[str, str | int | None]
    summary: str

## 4. OpenDART 조회 Node

이 Node가 실제 OpenDART API를 호출합니다. 반환된 딕셔너리는 `financial_data`라는 State 항목으로 합쳐집니다.

수치는 `client.py`가 반환한 원본 금액 문자열이며, 이 단계에서는 단위 변환이나 LLM 설명을 하지 않습니다.

In [ ]:
def fetch_financials(
    state: OpenDARTWorkflowState,
) -> dict[str, dict[str, str | int | None]]:
    financial_data = get_financial_summary(
        corp_name=state["company_name"],
        business_year=state["business_year"],
        api_key=api_key,
    )
    return {"financial_data": financial_data}

## 5. 요약 Node

이 Node는 API를 다시 호출하지 않습니다. 이전 Node가 State에 넣은 `financial_data`만 읽어 사람이 확인하기 쉬운 문자열을 만듭니다. 값이 `None`이면 `조회되지 않음`으로 표시합니다.

In [ ]:
def create_summary(state: OpenDARTWorkflowState) -> dict[str, str]:
    data = state["financial_data"]
    amount = lambda label: data[label] if data[label] is not None else "조회되지 않음"
    summary = (
        f'{data["기업명"]} {data["사업연도"]}년 재무정보\n'
        f'- 매출액: {amount("매출액")}\n'
        f'- 영업이익: {amount("영업이익")}\n'
        f'- 당기순이익: {amount("당기순이익")}'
    )
    return {"summary": summary}

## 6. Graph 만들기

이번 workflow의 순서는 고정되어 있습니다. LangGraph가 `fetch_financials`를 실행하고, 그 반환값을 합친 State를 `create_summary`에 전달합니다.

In [ ]:
workflow = StateGraph(OpenDARTWorkflowState)

workflow.add_node("fetch_financials", fetch_financials)
workflow.add_node("create_summary", create_summary)

workflow.add_edge(START, "fetch_financials")
workflow.add_edge("fetch_financials", "create_summary")
workflow.add_edge("create_summary", END)

app = workflow.compile()

## 7. Graph 구조 확인

Mermaid 텍스트에서 두 Node가 고정된 순서로 연결됐는지 확인합니다.

In [ ]:
print(app.get_graph().draw_mermaid())

## 8. 실제 기업 입력 State 준비

아래 기업명과 사업연도는 직접 바꿔 볼 수 있습니다. 기업명은 OpenDART에 등록된 정식 명칭과 정확히 일치해야 합니다.

In [ ]:
input_state: OpenDARTWorkflowState = {
    "company_name": "삼성전자",
    "business_year": 2025,
}

input_state

## 9. Graph 실행

이 셀을 실행하면 `fetch_financials` Node가 실제 OpenDART API를 호출합니다. 이후 `create_summary` Node가 받은 데이터를 State에 추가합니다.

In [ ]:
final_state = app.invoke(input_state)
final_state

## 10. Node 사이에 전달된 실제 데이터 확인

`financial_data`는 첫 번째 Node가 추가한 값이고, `summary`는 두 번째 Node가 추가한 값입니다. 둘을 나눠 확인합니다.

In [ ]:
final_state["financial_data"]

In [ ]:
print(final_state["summary"])

## 11. 최소 확인

이 확인은 정량 평가가 아니라, 실제 조회 결과가 State를 통해 다음 Node까지 전달됐는지 보는 학습용 점검입니다.

In [ ]:
assert final_state["financial_data"]["기업명"] == "삼성전자"
assert final_state["financial_data"]["사업연도"] == 2025
assert "매출액" in final_state["summary"]

print("실제 OpenDART 결과가 Graph State를 거쳐 요약 Node에 전달되었습니다.")

## 이번 단계에서 기억할 점

- LangGraph Node는 LLM이나 `@tool` 없이도 일반 Python 함수와 외부 API를 실행할 수 있습니다.
- `fetch_financials` Node는 실제 데이터를 `financial_data`에 추가합니다.
- `create_summary` Node는 API를 다시 호출하지 않고, 이전 Node가 추가한 State를 사용합니다.
- 이 단계의 수치는 원본 금액 문자열입니다. 단위 변환, 계산, LLM 답변 생성은 아직 하지 않습니다.